# 05 - End-to-End Orchestration

This notebook demonstrates the complete workflow: loading invoices, extracting data, validating, and submitting to Fakturoid.


In [2]:
from src.document_processor import DocumentProcessor
from src.ai_extractor import AIExtractor
from src.fakturoid_client import FakturoidClient
from src.config import config
from src.agent import InvoiceProcessingAgent
import json
from pathlib import Path


In [ ]:
# Initialize agent
agent = InvoiceProcessingAgent(config)

print("Invoice Processing Agent initialized")
print(f"Mode: {config.processing.mode}")
print(f"Auto-submit: {config.processing.auto_submit}")
print(f"Invoices directory: {config.directories.invoices}")



TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'Config'

In [ ]:
# Process all invoices (with manual approval)
print("="*60)
print("PROCESSING ALL INVOICES")
print("="*60)

results = agent.process_all_invoices()

print(f"\n{'='*60}")
print("PROCESSING SUMMARY")
print(f"{'='*60}")
print(f"Total invoices: {results['total']}")
print(f"Successful: {results['successful']}")
print(f"Failed: {results['failed']}")
print(f"Skipped: {results['skipped']}")


In [ ]:
# Show detailed results
print("\nDetailed Results:")
for result in results.get('details', []):
    print(f"\n{'-'*60}")
    print(f"File: {result['filename']}")
    print(f"Status: {result['status']}")
    
    if result['status'] == 'success':
        print(f"Invoice Number: {result.get('invoice_number', 'N/A')}")
        print(f"Amount: {result.get('amount', 'N/A')}")
        if result.get('fakturoid_id'):
            print(f"Fakturoid ID: {result['fakturoid_id']}")
    elif result['status'] == 'failed':
        print(f"Error: {result.get('error', 'Unknown error')}")
    
    if result.get('extracted_data'):
        print(f"\nExtracted Data:")
        print(json.dumps(result['extracted_data'], indent=2, ensure_ascii=False))


In [ ]:
# Process a single invoice with detailed steps
files = agent.doc_processor.list_invoice_files()

if files:
    test_file = files[0]
    print(f"Processing single invoice: {test_file.name}")
    print("="*60)
    
    # Step 1: Extract data
    print("\n1. Extracting data...")
    invoice_data = agent.ai_extractor.extract_invoice_data(test_file)
    print(json.dumps(invoice_data, indent=2, ensure_ascii=False))
    
    # Step 2: Validate
    print("\n2. Validating data...")
    is_valid, validation_errors = agent.validate_invoice_data(invoice_data)
    if is_valid:
        print("   ✓ Data is valid")
    else:
        print("   ✗ Validation errors:")
        for error in validation_errors:
            print(f"     - {error}")
    
    # Step 3: Submit (if valid and auto-submit is enabled)
    if is_valid and config.processing.auto_submit:
        print("\n3. Submitting to Fakturoid...")
        try:
            result = agent.fakturoid.create_expense_invoice(invoice_data)
            print(f"   ✓ Submitted successfully! ID: {result.get('id')}")
        except Exception as e:
            print(f"   ✗ Submission failed: {e}")
    else:
        print("\n3. Submission skipped (auto-submit disabled or validation failed)")
else:
    print("No invoice files found.")


In [ ]:
# Check processed invoices directory
processed_dir = config.directories.processed
processed_files = list(processed_dir.glob("*"))

print(f"Processed files directory: {processed_dir}")
print(f"Number of processed files: {len(processed_files)}")

if processed_files:
    print("\nProcessed files:")
    for f in processed_files[:10]:
        print(f"  - {f.name}")
    if len(processed_files) > 10:
        print(f"  ... and {len(processed_files) - 10} more")
